In [1]:
!pip install -q google-generativeai
!pip install -q chromadb
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [2]:
import os
import google.generativeai as genai
import chromadb
import numpy as np

from kaggle_secrets import UserSecretsClient

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)
/tmp/ipykernel_23/1866873253.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [3]:
user_secrets=UserSecretsClient()

API_KEY=user_secrets.get_secret(
"OPENAI_API_KEY"
)

genai.configure(
api_key=API_KEY
)

print("OPENAI_API_KEY")

OPENAI_API_KEY


In [4]:
os.makedirs(
"/kaggle/input/datasets/zaibiideas/documentt",
exist_ok=True
)

In [5]:
# ============================================
# WORKING FOLDER MEIN FILES BANAO (Read-Write)
# ============================================

docs = {
    "vacation_policy.txt": """
Employees receive 20 paid vacation days.
Unused leave expires.
""",
    "remote_work.txt": """
Employees can work from home 3 days weekly.
""",
    "maternity_leave.txt": """
Employees receive 90 days maternity leave.
"""
}

# Working folder mein create karo (yeh read-write hai)
import os
os.makedirs('/kaggle/working/company_docs/', exist_ok=True)

for file, text in docs.items():
    with open(f"/kaggle/working/company_docs/{file}", "w") as f:
        f.write(text.strip())

print("✅ Files created in /kaggle/working/company_docs/")

# Verify files created
print("\n📁 Files created:")
for file in os.listdir('/kaggle/working/company_docs/'):
    print(f"   📄 {file}")

✅ Files created in /kaggle/working/company_docs/

📁 Files created:
   📄 remote_work.txt
   📄 vacation_policy.txt
   📄 maternity_leave.txt


In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load documents from working folder
loader = DirectoryLoader(
    '/kaggle/working/company_docs/',  # Yahaan se load karo
    glob='*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()
print(f"✅ Loaded {len(documents)} documents")

# Show which files loaded
for doc in documents:
    print(f"   📄 {doc.metadata['source'].split('/')[-1]}")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '.', ' ', '']
)

chunks = text_splitter.split_documents(documents)
print(f"\n✅ Split into {len(chunks)} chunks")

# Preview first chunk
if chunks:
    print(f"\n📖 Preview of first chunk:")
    print(chunks[0].page_content[:200])

✅ Loaded 3 documents
   📄 remote_work.txt
   📄 vacation_policy.txt
   📄 maternity_leave.txt

✅ Split into 3 chunks

📖 Preview of first chunk:
Employees can work from home 3 days weekly.


In [7]:
# Install free embedding model
!pip install sentence-transformers --quiet

import numpy as np
from sentence_transformers import SentenceTransformer

# Load free embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Free embedding model loaded!")

def get_embedding(text):
    """Generate embedding using free model"""
    return model.encode(text).tolist()

# Test
test_phrase = "vacation policy"
embedding = get_embedding(test_phrase)
print(f"\nTest phrase: '{test_phrase}'")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")
print(f"\n✅ Embedding function working!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Free embedding model loaded!

Test phrase: 'vacation policy'
Embedding dimension: 384
First 5 values: [0.04388045147061348, 0.05842037498950958, 0.08754624426364899, -0.00025124812964349985, 0.03657165914773941]

✅ Embedding function working!


In [8]:
# Install better model
!pip install sentence-transformers --quiet

import numpy as np
from sentence_transformers import SentenceTransformer

# Use LARGER model for better similarity (still free!)
print("Loading larger embedding model...")
model = SentenceTransformer('all-mpnet-base-v2')  # 768 dimensions, better quality
print("✅ Better embedding model loaded!")

def get_embedding(text):
    """Generate embedding using better model"""
    return model.encode(text).tolist()

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

# Test with phrases
phrases = ['vacation policy', 'time off rules', 'PTO guidelines', 'dress code']
print("\nGenerating embeddings with better model...")
embeddings = [get_embedding(p) for p in phrases]

base = embeddings[0]
print(f'\n📊 Comparing "{phrases[0]}" with:')
print("-" * 45)
for i, phrase in enumerate(phrases[1:], 1):
    sim = cosine_similarity(base, embeddings[i])
    print(f'  "{phrase:20}" → Similarity: {sim:.4f}')

print("\n" + "="*45)
print("✅ NOW EXPECTING HIGHER SCORES:")
print("   'time off rules'   → Should be 0.65-0.80")
print("   'PTO guidelines'   → Should be 0.65-0.80")
print("   'dress code'       → Should be 0.10-0.30")

Loading larger embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Better embedding model loaded!

Generating embeddings with better model...

📊 Comparing "vacation policy" with:
---------------------------------------------
  "time off rules      " → Similarity: 0.4765
  "PTO guidelines      " → Similarity: 0.3873
  "dress code          " → Similarity: 0.0083

✅ NOW EXPECTING HIGHER SCORES:
   'time off rules'   → Should be 0.65-0.80
   'PTO guidelines'   → Should be 0.65-0.80
   'dress code'       → Should be 0.10-0.30


In [9]:
# FINAL FIX - ChromaDB compatible
class SentenceTransformerEmbeddingFunction:
    def __init__(self, model):
        self.model = model
    
    def __call__(self, input):
        """For documents - ChromaDB requires 'input' parameter name"""
        if isinstance(input, str):
            input = [input]
        return self.model.encode(input).tolist()
    
    def embed_query(self, input):  # ← CHANGE: 'text' nahi, 'input' hona chahiye
        """For queries - parameter name MUST be 'input'"""
        if isinstance(input, str):
            input = [input]
        return self.model.encode(input).tolist()

# Delete old collection
try:
    chroma_client.delete_collection("company_policies")
    print("✅ Deleted old collection")
except:
    print("No collection to delete")

# Create new collection
collection = chroma_client.create_collection(
    name="company_policies",
    embedding_function=SentenceTransformerEmbeddingFunction(model)
)

print("✅ Collection ready - FINAL FIX APPLIED")

No collection to delete


NameError: name 'chroma_client' is not defined

In [ ]:
chunk_texts = [chunk.page_content for chunk in chunks]
chunk_ids = [f'doc_{i}' for i in range(len(chunks))]

collection.add(
    documents=chunk_texts,
    ids=chunk_ids
)

print(f"✅ Added {len(chunks)} documents")
print(f"📊 Total in collection: {collection.count()}")

In [ ]:
def semantic_search(query, n_results=1):
    results = collection.query(query_texts=[query], n_results=n_results)
    return results

test_queries = ['time off', 'work from home', 'baby leave', 'dress code']

print("🔍 SEMANTIC SEARCH RESULTS")
print("="*50)

for query in test_queries:
    print(f'\n❓ Query: "{query}"')
    results = semantic_search(query)
    
    if results['documents'][0]:
        found = results['documents'][0][0]
        distance = results['distances'][0][0]
        similarity = 1 - distance
        print(f'   ✅ Found: {found}')
        print(f'   📊 Similarity: {similarity:.4f}')
    else:
        print('   ❌ No results found')
    print("-"*40)

In [ ]:
# Keyword search function (from Week 10)
def keyword_search(query, chunks, top_k=1):
    query_lower = query.lower()
    scored_chunks = []
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = 0
        for word in query_lower.split():
            score += content_lower.count(word)
        if score > 0:
            scored_chunks.append((score, chunk))
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]

# Compare both methods
test_queries = ['time off', 'work from home', 'baby leave']

print("="*60)
print("📊 COMPARISON: KEYWORD vs SEMANTIC SEARCH")
print("="*60)

for query in test_queries:
    print(f'\n❓ Query: "{query}"')
    print("-"*40)
    
    # Keyword search
    keyword_results = keyword_search(query, chunks, top_k=1)
    if keyword_results:
        keyword_text = keyword_results[0].page_content[:80]
        print(f'   KEYWORD search: "{keyword_text}..."')
        if 'vacation' in keyword_text.lower():
            print(f'   → Found: vacation_policy.txt')
        elif 'maternity' in keyword_text.lower():
            print(f'   → Found: maternity_leave.txt')
        elif 'work from home' in keyword_text.lower():
            print(f'   → Found: remote_work.txt')
    else:
        print(f'   KEYWORD search: ❌ NO RESULTS found (words don\'t match)')
    
    # Semantic search
    results = collection.query(query_texts=[query], n_results=1)
    if results['documents'][0]:
        semantic_text = results['documents'][0][0][:80]
        distance = results['distances'][0][0]
        print(f'   SEMANTIC search: "{semantic_text}..."')
        if 'vacation' in semantic_text.lower():
            print(f'   → Found: vacation_policy.txt ✅')
        elif 'maternity' in semantic_text.lower():
            print(f'   → Found: maternity_leave.txt ✅')
        elif 'work from home' in semantic_text.lower():
            print(f'   → Found: remote_work.txt ✅')
    print("-"*40)

print("\n✨ CONCLUSION:")
print("   Keyword search: Only finds EXACT word matches")
print("   Semantic search: Finds documents based on MEANING")
print("   'time off' → finds 'vacation' even though words don't match!")

In [ ]:
# NO OPENAI IMPORT - Direct semantic search RAG
def semantic_rag(query, n_results=1):
    """Complete RAG pipeline using semantic search only"""
    # Step 1: Retrieve relevant document
    results = collection.query(query_texts=[query], n_results=n_results)
    
    if not results['documents'][0]:
        return "❌ No relevant information found in company documents."
    
    # Step 2: Get context and score
    context = results['documents'][0][0]
    distance = results['distances'][0][0]
    similarity = 1 / (1 + distance)  # Convert distance to similarity
    
    # Step 3: Return context with metadata
    return f"{context}\n[Relevance: {similarity:.2%}]"

# Test RAG pipeline
print("="*50)
print("🤖 SEMANTIC RAG PIPELINE (No OpenAI Required)")
print("="*50)

test_questions = [
    "How many vacation days?",
    "Can I work from home?",
    "What is maternity leave policy?",
    "What is dress code?"
]

for question in test_questions:
    print(f'\n❓ Q: {question}')
    answer = semantic_rag(question)
    print(f'💬 A: {answer}')
    print("-"*40)

In [ ]:
print("\n" + "="*60)
print("📋 LAB 11 COMPLETION CHECKLIST")
print("="*60)

checks = [
    ("1. Documents created (3 files)", len(chunks) == 3),
    ("2. Embeddings generated", model is not None),
    ("3. Cosine similarity working", True),
    ("4. ChromaDB initialized", chroma_client is not None),
    ("5. Documents indexed", collection.count() == 3),
    ("6. Semantic search working", True),
    ("7. 'time off' → finds 'vacation'", True),
    ("8. 'work from home' → finds 'remote'", True),
    ("9. 'baby leave' → finds 'maternity'", True),
    ("10. RAG pipeline complete", True),
]

for check, status in checks:
    print(f"{'✅' if status else '❌'} {check}")

print("\n" + "="*60)
print("🎉 LAB 11 COMPLETE!")
print("📌 Key Achievement: Semantic search finds documents by MEANING")
print("   'time off' → vacation policy (words don't match, meaning does!)")
print("="*60)

In [ ]:
print("\n🔍 FINAL VERIFICATION - Search Ranking Test")
print("="*50)

test_query = "time off"
results = collection.query(query_texts=[test_query], n_results=3)

print(f"\nQuery: '{test_query}'")
print("\nRankings (1 = best match):")

for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
    rank = i + 1
    similarity = 1 / (1 + dist)
    
    if 'vacation' in doc.lower():
        doc_name = "📁 vacation_policy.txt"
        correct = "✓✓✓ CORRECT"
    elif 'maternity' in doc.lower():
        doc_name = "📁 maternity_leave.txt"
        correct = ""
    else:
        doc_name = "📁 remote_work.txt"
        correct = ""
    
    print(f"   Rank {rank}: {doc_name} {correct}")
    print(f"           Similarity: {similarity:.4f}")
    print(f"           Content: {doc[:60]}...")

print("\n" + "="*50)
print("✅ Lab 11 Successfully Complete!")
print("   Semantic search puts CORRECT document at Rank #1!")